In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import scipy.constants as phy_const

PP = np.loadtxt('PPP.txt')
PP_matrix = np.matrix(PP)

values = np.loadtxt('values.txt')
values_matrix = np.matrix(values)
print(np.shape(PP_matrix))

In [12]:
ng   = PP[0, :]
ni   = PP[1, :]
vi   = PP[2, :]
Te   = PP[3, :]
ve   = PP[4, :]
Ue_y = PP[5, :]

Barr = values[0, :]
x_center = values[1, :]
alpha_B = values[2, :]
wce = phy_const.e*Barr/phy_const.m_e
nu_m = alpha_B*wce

In [ ]:
plt.figure()
# plt.plot(ni, label='ni')
# plt.plot(vi, label='vi')
plt.plot(ni, label='Te')


In [14]:
Rei = - (phy_const.m_e * ni * nu_m * Ue_y)
Bforce = phy_const.e * ni * Barr * ve
flux = phy_const.m_e * ni * ve * Ue_y
flux_der = -np.gradient(flux, x_center)
somma = Rei + Bforce + flux_der

In [ ]:
plt.figure()
plt.plot(Rei, label='Rei')
plt.plot(Bforce, label='Bforce')
plt.plot(flux_der, label='flux')
plt.plot(somma, label='somma', marker ="v")
plt.legend()
plt.axhline(y=0, color='k', linestyle='--')

In [16]:
empirical_term = np.loadtxt('empirical_term.txt')

empirical_term_interp = np.interp(x_center, empirical_term[:, 0]/100, empirical_term[:, 1])


In [ ]:
plt.figure()
plt.plot(empirical_term[:, 0]/100, empirical_term[:, 1])
plt.plot(x_center, empirical_term_interp, label='empirical_term')


In [29]:
fBarr, ni, div_uey = np.loadtxt("bottom_values.txt")


In [ ]:
plt.plot(fBarr / ni - phy_const.electron_mass * div_uey / (phy_const.e * ni), marker = "v", label = "1")
plt.plot(-phy_const.electron_mass * div_uey / (phy_const.e * ni), marker = "v", label = "2")
plt.plot(fBarr / ni, marker = "v", label = "3")
plt.legend()
Term_2 = fBarr / ni - phy_const.electron_mass * div_uey / (phy_const.e * ni)
# print("2 Term_2: ", Term_2[-4:])
# print("2 fBarrr: ", (fBarr / ni)[-4:])
# print("2 div_uey: ", (div_uey / (phy_const.e * ni))[-4:])
from scipy import integrate
value_simpson_2 = integrate.simpson(Term_2 , x=x_center)
print(value_simpson_2)

plt.figure()
plt.plot(div_uey)

In [ ]:

def linear_extrapolation_multi(vec, num_points=3):
    # Use at least two points for linear extrapolation
    if len(vec) < 2:
        raise ValueError("Vector must have at least two elements for extrapolation.")
    
    # Ensure num_points is not greater than the vector length
    num_points = min(num_points, len(vec) - 1)
    
    # Fit a line (degree 1 polynomial) to the first few points
    start_fit = np.polyfit(range(num_points), vec[:num_points], 1)
    # Predict the value before the first point using the line
    start_extrapolated_value = np.polyval(start_fit, -1)  # x = -1 for extrapolation one step back

    # Fit a line to the last few points
    end_fit = np.polyfit(range(len(vec) - num_points, len(vec)), vec[-num_points:], 1)
    # Predict the value after the last point using the line
    end_extrapolated_value = np.polyval(end_fit, len(vec))  # x = len(vec) for extrapolation one step forward
    
    # Add the extrapolated values to the vector
    extended_vec = np.insert(vec, 0, start_extrapolated_value)  # Insert at the beginning
    extended_vec = np.append(extended_vec, end_extrapolated_value)  # Append at the end
    
    return extended_vec

empirical_term = np.loadtxt('empirical_term_add.txt')
# empirical_term = np.loadtxt('empirical_term_1.txt')
empirical_term_interp_y = np.interp(x_center, empirical_term[:, 0]/100, empirical_term[:, 1])
print("Empirical term y loaded.")
empirical_term_interp_x = np.interp(x_center, empirical_term[:, 0]/100, empirical_term[:, 2])
print("Empirical term x loaded.")
tau_xy = np.interp(x_center, empirical_term[:, 0]/100, empirical_term[:, 3])
tau_xy_aug = linear_extrapolation_multi(tau_xy, 2)
deltaX = x_center[1] - x_center[0]
x_center_aug = np.concatenate([[x_center[0] - deltaX], x_center, [x_center[-1] + deltaX]], axis=0)

In [ ]:
plt.plot(x_center, tau_xy)
plt.plot(x_center_aug, tau_xy_aug, "--")
plt.xlim(0,0.002)
# plt.xlim(2.45e-2,2.5e-2)
